###Read files using data stream read API

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType
from pyspark.sql import functions as F

customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("telephone", StringType(), True),
    StructField("email", StringType(), True),
    StructField("member_since", DateType(), True),
    StructField("created_timestamp", TimestampType(), True)
])

In [0]:
df = (spark.readStream.format("JSON")
      .schema(customers_schema)
      .load("/Volumes/gizmobox/landing/operational_data/Customer_Stream/"))

In [0]:
df_metadata = df.withColumn("file_path", F.col("_metadata.file_path"))\
                .withColumn("ingestion_date", F.current_timestamp())
                

###Write 

In [0]:
streaming_query = (df_metadata.writeStream\
    .format("Delta")\
    .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/Customer_Stream/_checkpoint_stream")\
    .toTable("gizmobox.bronze.customers_stream"))

In [0]:
streaming_query.stop()